In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.decomposition import PCA
import sys
import argparse
import os
import glob
import tqdm

import utils_report as ru
from utils_sensors import get_sensor_indices_from_cfg

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import rcParams
import seaborn as sns
import statsmodels.api as sm

import scipy

import importlib
import utils_psd
importlib.reload(utils_psd)  # force reload every time

from utils_psd import ( 
    mark_multiple_event_windows,
    add_spectrogram_power_column,
    plot_spectrogram_for_example_episode,
    plot_stacked_psds_for_interacting_pair,
    plot_power_around_event,   
    plot_power_by_boolean_feature,
    add_spectrogram_similarity_with_nearest_agent_column,
)

from utils_pairwise import (
    get_interactions_df_Nfish,
    merge_pairwise_interaction_timestep_df_with_dff,
)

import time
from cfg import ENV_PARAMS

fps = ENV_PARAMS["fps_sim"]  # samples per second
dt = 1.0 / fps  # seconds per time step

import scipy

rcParams["pdf.fonttype"] = 42  # Use Type 42 (TrueType) fonts to save text as text
rcParams["ps.fonttype"] = 42  # For PostScript as well, if needed
import seaborn as sns

# print("numpy version:", np.__version__)
# print("pandas version:", pd.__version__)

## Load data

In [ ]:
# flat_pkl_file = "results/rmappo-MultiAgentFishEnv-3MSensorDropout5xGRU/20250721_162614/outputs/MAFish_neural_20250721_162614_07PhNBro_agg_flattened.pkl"
# flat_pkl_file = "results/rmappo-MultiAgentFishEnv-3MSensorDropout5xGRU/20250721_162614/outputs/MAFish_neural_20250721_162614_07PhNBro_agg_flattened.pkl"

flat_pkl_file = "results/rmappo-MultiAgentFishEnv-3MSensorDropout5xRNN/20250730_033343/outputs/MAFish_neural_20250730_033343_TwoFishLongRangeforagingSeed1_agg_flattened.pkl"

# task= "foraging"  # Default task
task = "2ff"  # Pairwise only 


In [ ]:

parser = argparse.ArgumentParser()
parser.add_argument("--flat_pkl_file", type=str, help="Path to input pickle file")
parser.add_argument(
    "--model_seed", type=int, default=137, help="Random seed for classifier"
)
parser.add_argument(
    "--name_mode",
    type=str,
    default="name",
    choices=["name", "short_name"],
    help="Feature label mode",
)
parser.add_argument(
    "--task",
    type=str,
    default="foraging",
    help="Task e.g., 'foraging', '2fip', etc.",
)
args = parser.parse_args()

# Check if we're in interactive mode or batch mode
batchmode = False
if "ipykernel_launcher" in sys.argv[0]:
    print("Interactive mode")

    # High res figures 
    from IPython.display import set_matplotlib_formats
    set_matplotlib_formats('retina')
    plt.rcParams['figure.dpi'] = 300 

else:
    batchmode = True
    print("Batch/CLI mode")
    # Parses the command line arguments below

if batchmode:
    flat_pkl_file = args.flat_pkl_file
    model_seed = args.model_seed
    task = args.task
    name_mode = args.name_mode
else:
    # fallback for interactive mode
    model_seed = args.model_seed
    name_mode = args.name_mode


In [ ]:

dff, OUTPUTS_FOLDER, pkl_str = ru.load_flat_pkl_file(flat_pkl_file, task=task)
dff.columns

In [ ]:
# dff["distance_to_nearest_agent"].hist(bins=100)
# plt.show()

# # Bin distance_to_nearest_agent into bins with cutoffs at bin_width cm increments
# bin_width = 5 # cm
# num_bins = int(dff["distance_to_nearest_agent"].max() // bin_width) + 1
# print(f"{num_bins} bins with width {bin_width} cm")
# dff[f"distance_to_nearest_agent_{bin_width}cm_bin"] = dff["distance_to_nearest_agent"].apply(lambda x: math.floor(x / bin_width) * bin_width)
# dff[f"distance_to_nearest_agent_{bin_width}cm_bin"].hist(bins=num_bins)


dff["distance_to_nearest_agent"].hist(bins=100)
plt.show()

# Bin distance_to_nearest_agent into bins with cutoffs at bin_width cm increments
bin_width = 5 # cm
num_bins = int(dff["distance_to_nearest_agent"].max() // bin_width) + 1
print(f"{num_bins} bins with width {bin_width} cm")
dff[f"distance_to_nearest_agent_binned"] = dff["distance_to_nearest_agent"].apply(lambda x: (math.floor(x / bin_width) + 1) * bin_width)
dff[f"distance_to_nearest_agent_binned"].hist(bins=num_bins)

dff[f"distance_to_nearest_agent_binned"].value_counts().sort_index()


# RNN Spectral/Modal Analysis

## PSD analysis without features

In [ ]:


dff = add_spectrogram_power_column(dff)


In [ ]:

dff["spectrogram_power"].head(n=20)

In [ ]:
# plot_spectrogram_for_example_episode(dff, env_id=0, episode_index=0, agent_id=0)

In [ ]:
# env_id, episode_index, agent_pair = 1, 0, (0, 1)  # Good example
# plot_stacked_psds_for_interacting_pair(dff, env_id, episode_index, agent_pair)

In [ ]:
dff = add_spectrogram_similarity_with_nearest_agent_column(dff, metric="cosine")
dff["spectrogram_similarity_with_nearest_agent"].head(n=20)

In [ ]:
# def plot_wideband_power_by_distance_bin(dff, power_column="mean_wideband_power"):
#     """
#     Plot the mean wideband power for each distance bin with standard deviation error bars.
#     """
#     # Group by distance bin and compute mean and standard deviation
#     grouped = dff.groupby("distance_to_nearest_agent_10cm_bin")["mean_wideband_power"].agg(['mean', 'std']).reset_index()
#     grouped = grouped.rename(columns={"mean": "mean_wideband_power", "std": "std_wideband_power"})

#     # Plot
#     plt.figure(figsize=(10, 6))
#     ax = sns.barplot(
#         x="distance_to_nearest_agent_10cm_bin", 
#         y="mean_wideband_power", 
#         data=grouped, 
#         ci=None,
#         color='skyblue'
#     )
    
#     # Add error bars manually
#     ax.errorbar(
#         x=range(len(grouped)),
#         y=grouped["mean_wideband_power"],
#         yerr=grouped["std_wideband_power"],
#         fmt='none',
#         c='black',
#         capsize=5
#     )

#     plt.xlabel("Distance to Nearest Agent (10 cm bins)")
#     plt.ylabel("Mean Wideband Power")
#     plt.title("Mean Wideband Power by Distance to Nearest Agent")
#     plt.xticks(rotation=45)
#     plt.tight_layout()
#     plt.show()

# plot_wideband_power_by_distance_bin(dff)


In [ ]:

def plot_feature_by_distance_bin(dff, feature_col, bin_col="distance_to_nearest_agent_binned", plot_type="bar"):
    """
    Plot the distribution or mean+std of a specified feature across distance bins.

    Parameters:
    - dff: DataFrame
    - feature_col: str, column to aggregate and plot
    - bin_col: str, the column to use for binning (default: distance bin)
    - plot_type: str, "bar" or "box" (default: "bar")
    """
    # Mapping for pretty labels
    label_map = {
        "mean_wideband_power": "Mean Wideband Power",
        "spectrogram_similarity_with_nearest_agent": "Spectrogram Similarity with Nearest Agent"
    }

    y_label = label_map.get(feature_col, feature_col.replace("_", " ").title())
    title = f"{y_label} by Distance to Nearest Agent"

    plt.figure(figsize=(6, 4))

    if plot_type == "bar":
        # Group by bin and aggregate mean/std
        grouped = dff.groupby(bin_col)[feature_col].agg(['mean', 'std']).reset_index()
        grouped = grouped.rename(columns={"mean": "mean_val", "std": "std_val"})

        ax = sns.barplot(
            x=bin_col,
            y="mean_val",
            data=grouped,
            ci=None,
            color="skyblue"
        )
        ax.errorbar(
            x=range(len(grouped)),
            y=grouped["mean_val"],
            yerr=grouped["std_val"],
            fmt='none',
            c='black',
            capsize=5
        )
    elif plot_type == "box":
        ax = sns.boxplot(
            x=bin_col,
            y=feature_col,
            data=dff,
            color="skyblue"
        )
    else:
        raise ValueError(f"Unsupported plot_type: {plot_type}. Use 'bar' or 'box'.")

    plt.xlabel("Distance to Nearest Agent (10 cm bins)")
    plt.ylabel(y_label)
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_feature_by_distance_bin(dff, "mean_wideband_power", plot_type="bar")
plot_feature_by_distance_bin(dff, "mean_wideband_power", plot_type="box")
plot_feature_by_distance_bin(dff, "spectrogram_similarity_with_nearest_agent", plot_type="bar")
plot_feature_by_distance_bin(dff, "spectrogram_similarity_with_nearest_agent", plot_type="box")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import numpy as np
import pandas as pd

def plot_regression_against_distance(dff, feature_col, distance_col="distance_to_nearest_agent", samples=None):
    """
    Fit and plot a linear regression of a feature against distance to nearest agent.
    
    Parameters:
    - dff: DataFrame
    - feature_col: str, the dependent variable
    - distance_col: str, the independent variable (default: 'distance_to_nearest_agent')
    """
    # Drop NaNs
    df = dff[[distance_col, feature_col]].dropna()
    if samples is not None:
        df = df.sample(n=samples, random_state=42)
    
    X = df[distance_col]
    y = df[feature_col]

    # Add constant for intercept
    X_const = sm.add_constant(X)

    # Fit linear regression
    model = sm.OLS(y, X_const).fit()
    print(model.summary())

    # Report coefficients and p-values
    coef = model.params[1]
    pval = model.pvalues[1]
    r2 = model.rsquared
    # print(f"\nCoefficient: {coef:.4f}, p-value: {pval:.4g}, R² = {r2:.4f}")
    annotation_text = f"Slope: {coef:.4f} p-val: {pval:.4g} $R^2$: {r2:.3f}"

    # Predict over sorted values for smooth curve
    x_pred = np.linspace(X.min(), X.max(), 100)
    x_pred_const = sm.add_constant(x_pred)
    y_pred = model.predict(x_pred_const)

    # Get confidence intervals
    pred_summary = model.get_prediction(x_pred_const).summary_frame(alpha=0.05)
    lower = pred_summary['mean_ci_lower']
    upper = pred_summary['mean_ci_upper']

    # Plot
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=X, y=y, s=2, alpha=0.2, label="Data", color="gray")
    plt.plot(x_pred, y_pred, color="blue", label="Linear Fit")
    plt.fill_between(x_pred, lower, upper, color="lightblue", alpha=0.8, label="95% CI")
    
    # Labels and stats
    plt.xlabel("Distance to Nearest Agent (cm)")
    ylabel_map = {
        "mean_wideband_power": "Mean Wideband Power",
        "spectrogram_similarity_with_nearest_agent": "Spectrogram Similarity with Nearest Agent"
    }
    plt.ylabel(ylabel_map.get(feature_col, feature_col.replace("_", " ").title()))
    # plt.title(f"Linear Regression: {feature_col} ~ distance_to_nearest_agent")

    # Annotate stats in upper-left corner
    plt.gca().text(
        # 0.02, 0.98,
        0.5, 0.02, # centered at bottom
        annotation_text,
        transform=plt.gca().transAxes,
        verticalalignment="bottom",
        horizontalalignment="center",
        fontsize=9,
        # bbox=dict(facecolor="white", edgecolor="gray", boxstyle="round,pad=0.3")
    )

    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.show()


plot_regression_against_distance(dff, "mean_wideband_power", samples=10000)
plot_regression_against_distance(dff, "spectrogram_similarity_with_nearest_agent", samples=10000)


## Interactions based alternative pipeline

In [ ]:
import utils_pairwise
importlib.reload(utils_pairwise)  # force reload every time
from utils_pairwise import get_interactions_df_Nfish

interactions_df, timestep_df = get_interactions_df_Nfish(dff, theta_close=np.pi/4, theta_opposite=3*np.pi/4)

print(interactions_df["dominant_interaction_class"].value_counts())
print(interactions_df["dominant_interaction_class_filtered"].value_counts())



In [ ]:
print( set(dff.columns).intersection(set(timestep_df.columns)) )

print( set(timestep_df.columns) - set(dff.columns))


In [ ]:

dff_interactions = merge_pairwise_interaction_timestep_df_with_dff(dff, timestep_df)
print(dff_interactions.shape, dff.shape)
print(set(dff_interactions.columns) - set(dff.columns))
print(dff_interactions[["has_nearby", "interaction_class"]].value_counts(dropna=False))

# dff.duplicated(subset=["env_id", "episode_index", "time_step", "agent_id"]).sum()
dff_interactions[["has_nearby", "interaction_class"]]
print(dff_interactions[["has_nearby", "interaction_class"]].value_counts(dropna=False))
# print(pd.crosstab(dff_interactions["has_nearby"], dff_interactions["interaction_class"], dropna=False))

def plot_power_by_nearby_and_interaction(dff_interactions, power_column="mean_wideband_power"):

    # Create group labels
    dff_plot = dff_interactions.copy()
    dff_plot["group"] = dff_plot["has_nearby"].astype(str) + " | " + dff_plot["interaction_class"].astype(str)

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=dff_plot, x="group", y=power_column)
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("has_nearby | interaction_class")
    plt.ylabel(power_column.replace("_", " ").title())
    plt.title(f"{power_column.replace('_', ' ').title()} by has_nearby × interaction_class")
    plt.tight_layout()
    plt.show()
    
plot_power_by_nearby_and_interaction(dff_interactions, power_column="mean_wideband_power")


In [ ]:
dff_interactions.columns

plot_power_by_nearby_and_interaction(dff_interactions, power_column="spectrogram_similarity_with_nearest_agent")




## Events/Features

In [ ]:
print(dff["eating_event"].value_counts())
print(dff["spectrogram_power"].isna().mean())
print(dff["has_nearby"].value_counts())
print(dff["has_nearby_emit_eod"].value_counts())

print(dff.columns)

In [ ]:
# plot_power_around_event(dff, event_column="eating_event", event_name="Eating", window_size=100)


In [ ]:
importlib.reload(utils_psd)  # force reload every time
from utils_psd import plot_power_by_boolean_feature

# plot_power_by_boolean_feature(dff, feature_column="has_nearby", feature_name=None, power_column="mean_wideband_power")
for feature_column in ["has_nearby", "has_nearby_emit_eod", "eating_event"]:
    plot_power_by_boolean_feature(dff, feature_column=feature_column, feature_name=None, power_column="mean_wideband_power")


In [ ]:

from utils_psd import plot_per_freq_band_violin_plots
df_stats = plot_per_freq_band_violin_plots(dff)
# df_stats.sort_values(by="Effect Size", ascending=False)


In [ ]:
# Deprecated -- use add_spectrogram_similarity_with_nearest_agent_column
# importlib.reload(utils_psd)  # force reload every time
# from utils_psd import compute_psd_similarity_between_agents
# compute_psd_similarity_between_agents(dff, metric="cosine")
# compute_psd_similarity_between_agents(dff, metric="euclidean")


In [ ]:

# from utils_psd import compute_cosine_similarity_between_multiple_agents # Probably broken, not used
# compute_cosine_similarity_between_multiple_agents(dff) # Probably broken, not used

In [ ]:

# from utils_psd import plot_cosine_similarity_timeseries # Probably broken, not used
# plot_cosine_similarity_timeseries(dff, window=10)


In [ ]:
from utils_psd import (    compute_baseline_spectrogram,
    plot_avg_spectrogram_by_condition,
    mark_multiple_event_windows,
)


dff = dff.sort_values(by=["env_id", "agent_id", "episode_index", "time_step"]).copy()
event_cols = [
    "has_nearby",
    # "has_nearby_emit_eod",
    # "was_bitten",
    # "bite_other_fish",
    # "eating_event",
    # "collided",
]


# Add per-event masks and control
dff = mark_multiple_event_windows(dff, event_cols, W=5)

# fs = ENV_PARAMS["fps_sim"]  # Hz

# Baseline (no mask)
freqs_base, power_base = compute_baseline_spectrogram(dff, fs=ENV_PARAMS["fps_sim"])

# Spectrogram by condition
spectra = plot_avg_spectrogram_by_condition(
    dff, event_cols=event_cols, W=5, fs=ENV_PARAMS["fps_sim"], show=True, delay=5,
)

plt.figure(figsize=(8, 4))
plt.plot(
    freqs_base, power_base, label="baseline (all data)", linestyle="--", color="gray"
)
# Overlay the event conditions
for label, (f, p) in spectra.items():
    plt.plot(f, p, label=label)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Average Spectral Power")
plt.title("Spectrogram Comparison: Events vs Baseline")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()